This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [1]:
#!pip install keras keras-hub --upgrade -q

In [2]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [3]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

## Language models and **the Transformer**

이 장에서는 다음 내용을 다룹니다.

* 딥러닝 모델을 이용한 텍스트 생성 방법
* 영어를 스페인어로 번역하는 모델 학습
* 텍스트 모델링 문제를 위한 강력한 아키텍처, 트랜스포머

이전 장에서 텍스트 전처리 및 모델링의 기초를 다룬 후, 이 장에서는 기계 번역과 같은 좀 더 복잡한 언어 문제를 다룹니다. ChatGPT와 같은 제품에 적용되어 자연어 처리(NLP) 분야에 대한 투자를 촉발시킨 트랜스포머 모델에 대한 탄탄한 이해를 쌓아갈 것입니다.

#### English-to-Spanish translation

우리는 영어-스페인어 번역 데이터셋을 사용할 것입니다. 다운로드해 봅시다.

In [5]:
import keras
import pathlib

zip_path = keras.utils.get_file(
    origin=(
        "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
    ),
    fname="spa-eng",
    extract=True,
)
text_path = pathlib.Path(zip_path) / "spa-eng" / "spa.txt"

텍스트 파일에는 각 줄마다 하나의 예시가 있습니다. 영어 문장 다음에 탭 문자가 오고, 그 다음에 해당 스페인어 문장이 옵니다. 이 파일을 분석해 보겠습니다.

In [6]:
with open(text_path, encoding="utf-8") as f:
    lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
    english, spanish = line.split("\t")
    spanish = "[start] " + spanish + " [end]"
    text_pairs.append((english, spanish))

저희의 텍스트 쌍은 다음과 같습니다.

In [7]:
import random
random.choice(text_pairs)

('He made her cry.', '[start] Le hizo llorar. [end]')

이제 이 데이터셋들을 섞어서 일반적인 학습, 검증, 테스트 세트로 나누어 보겠습니다.

In [8]:
import random

random.shuffle(text_pairs)
val_samples = int(0.15 * len(text_pairs))
train_samples = len(text_pairs) - 2 * val_samples
train_pairs = text_pairs[:train_samples]
val_pairs = text_pairs[train_samples : train_samples + val_samples]
test_pairs = text_pairs[train_samples + val_samples :]

다음으로, 영어와 스페인어 각각에 대한 두 개의 별도 텍스트 벡터화 레이어를 준비해 보겠습니다. 문자열 전처리 방식을 사용자 정의해야 합니다.

* 삽입한 "[start]"와 "[end]" 토큰을 유지해야 합니다. 기본적으로 [ ] 문자는 제거되지만, "start"라는 단어와 시작 토큰 "[start]"를 구분하기 위해 이 문자들을 남겨두어야 합니다.
* 언어마다 구두점 표기법이 다릅니다! 스페인어 텍스트 벡터화 레이어에서 구두점을 제거하려면 ¿ 문자도 함께 제거해야 합니다.

참고로, 실제 번역 모델에서는 구두점을 제거하는 대신 별도의 토큰으로 처리하여 구두점이 있는 문장도 생성할 수 있도록 해야 합니다. 하지만 여기서는 간단하게 모든 구두점을 제거하겠습니다.

In [11]:
from keras import layers
import tensorflow as tf

import string
import re

strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", ""
    )

vocab_size = 15000
sequence_length = 20

english_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)
spanish_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)
train_english_texts = [pair[0] for pair in train_pairs]
train_spanish_texts = [pair[1] for pair in train_pairs]
english_tokenizer.adapt(train_english_texts)
spanish_tokenizer.adapt(train_spanish_texts)

마지막으로, 데이터를 `tf.data` 파이프라인으로 변환할 수 있습니다. 이 파이프라인은 `inputs`와 `spanish` 두 개의 키를 가진 딕셔너리 `(inputs, target, sample_weights)` 튜플을 반환하도록 설계되었습니다. `inputs`는 토큰화된 영어 문장 `english`와 `spanish`를 포함하는 딕셔너리이고, `target`은 한 단계 앞선 스페인어 문장의 오프셋 값입니다. `sample_weights`는 Keras에게 손실과 메트릭을 계산할 때 사용할 레이블을 지정하는 데 사용됩니다. 출력 번역문의 길이는 모두 같지 않으며, 일부 레이블 시퀀스는 0으로 채워집니다. 우리는 실제 번역된 텍스트를 나타내는 0이 아닌 레이블에 대한 예측만 중요하게 생각합니다.

이는 방금 구축한 생성 모델에서 설정한 "오프 바이 원(off by one)" 레이블과 동일하며, 고정된 인코더 입력이 추가된 것입니다. 인코더 입력은 모델에서 별도로 처리됩니다.

In [12]:
batch_size = 64

def format_dataset(eng, spa):
    eng = english_tokenizer(eng)
    spa = spanish_tokenizer(spa)
    features = {"english": eng, "spanish": spa[:, :-1]}
    labels = spa[:, 1:]
    sample_weights = labels != 0
    return features, labels, sample_weights

def make_dataset(pairs):
    eng_texts, spa_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    spa_texts = list(spa_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, spa_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.shuffle(2048).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

다음은 저희 데이터셋 출력 결과입니다.

In [13]:
inputs, targets, sample_weights = next(iter(train_ds))
print(inputs["english"].shape)

(64, 20)


In [14]:
print(inputs["spanish"].shape)

(64, 20)


In [15]:
print(targets.shape)

(64, 20)


In [16]:
print(sample_weights.shape)

(64, 20)


이제 데이터가 준비되었으니, 모델을 구축할 차례입니다.

### The Transformer architecture

2017년, Vaswani 외 연구진은 획기적인 논문 "Attention Is All You Need"[1]에서 Transformer 아키텍처를 소개했습니다. 저자들은 우리가 방금 구축한 것과 같은 번역 시스템을 연구하고 있었는데, 핵심적인 발견은 논문 제목에 담겨 있습니다. 바로 어텐션(attention)이라는 간단한 메커니즘을 사용하여 순환 레이어(recurrent layer)를 전혀 사용하지 않고도 강력한 시퀀스 모델을 구축할 수 있다는 것입니다. 어텐션 개념 자체는 새로운 것이 아니었고, 논문 발표 당시 이미 몇 년 동안 자연어 처리 시스템에서 사용되고 있었습니다. 하지만 어텐션이 시퀀스를 통해 정보를 전달하는 데 필요한 유일한 메커니즘이 될 수 있을 정도로 유용하다는 사실은 당시로서는 매우 놀라운 발견이었습니다.

이 발견은 자연어 처리 분야는 물론 그 너머까지 혁명적인 변화를 가져왔습니다. 어텐션은 딥러닝에서 가장 영향력 있는 개념 중 하나로 빠르게 자리 잡았습니다. 이 섹션에서는 어텐션의 작동 방식과 시퀀스 모델링에 왜 그렇게 효과적인지 자세히 설명합니다. 그런 다음 어텐션을 사용하여 영어-스페인어 번역 모델을 다시 구축해 보겠습니다.

그렇다면, 지금까지 설명한 내용을 바탕으로 어텐션이란 정확히 무엇일까요? 그렇다면 어텐션은 지금까지 사용해 온 순환 신경망(RNN)을 어떻게 대체할 수 있을까요?

어텐션은 사실 방금 만든 RNN 모델과 같은 기존 RNN 모델을 강화하기 위해 개발되었습니다. 연구자들은 RNN이 주변 영역 내의 의존성을 모델링하는 데는 탁월하지만, 시퀀스 길이가 길어질수록 재현율이 떨어지는 것을 발견했습니다. 예를 들어, 문서에 대한 질문에 답하는 시스템을 구축한다고 가정해 보겠습니다. 문서 길이가 너무 길어지면 RNN의 결과는 인간의 예측 능력과는 비교할 수 없을 정도로 형편없어집니다.

이 책을 활용하여 날씨 예측 모델을 만든다고 생각해 보세요. 시간이 충분하다면 책 전체를 처음부터 끝까지 읽겠지만, 실제로 모델을 구현할 때는 시계열 관련 장에 특히 집중할 것입니다. 같은 장 안에서도 자주 참고할 특정 코드 예제나 설명이 있을 것입니다. 반면에 코드를 작성할 때는 이미지 컨볼루션과 같은 세부적인 내용에는 그다지 신경 쓰지 않을 것입니다. 이 책의 전체 단어 수는 10만 단어를 훨씬 넘는데, 이는 우리가 지금까지 다뤄본 어떤 시퀀스 길이보다 훨씬 깁니다. 하지만 인간은 텍스트에서 정보를 추출할 때 선택적이고 맥락적인 방식을 사용할 수 있습니다.

반면, RNN은 시퀀스의 이전 부분을 직접 참조할 수 있는 메커니즘이 없습니다. 모든 정보는 설계상 RNN 셀의 내부 상태를 거쳐 시퀀스의 모든 위치를 순환적으로 통과해야 합니다. 마치 이 책을 다 읽고 덮은 다음, 날씨 예측 모델을 완전히 기억에 의존해서 구현하려는 것과 같습니다. 어텐션 메커니즘은 신경망이 현재 처리 중인 입력에 따라 시퀀스의 특정 부분에 더 많은 가중치를 부여하고 다른 부분에는 더 적은 가중치를 부여할 수 있도록 하는 메커니즘을 구축하는 것입니다(그림 15.3).

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch15/attention-concept.fde57742.png" width="600"><br>Figure 15.3: The general concept of attention in deep learning: input features get assigned attention scores, which can be used to inform the next representation of the input.</p>


아인슈타인 합 표기법이란 무엇일까요?

머신러닝 코드에서 `np.einsum('ij,jk->ik', a, b)`와 같은 작은 코드 조각을 자주 볼 수 있습니다. 이것은 아인슈타인 합 표기법(Einstein summation notation)의 줄임말로, 아인슈타인 합 표기법이라고 합니다. 이 표기법을 익히면 복잡한 배열 연산을 명확하게 표현하는 데 유용하게 사용할 수 있습니다. 특히 Transformer 코드에서 자주 사용되는 이유입니다.

아인슈타인 합 표기법의 핵심 아이디어는 입력의 각 축을 고유한 문자로 나타내는 것입니다. 예를 들어, 3차 입력은 `ijk`로 표현할 수 있습니다. 그런 다음, 입력 개수는 상관없고 출력은 `input1,input2->output`과 같이 하나의 축을 갖는 표기법을 작성합니다. 이 표기법의 규칙은 다음과 같습니다.

입력에 동일한 문자가 있는 경우, 해당 축의 값을 서로 곱합니다. 이때 두 축의 크기는 같아야 합니다.

입력에는 있지만 출력에는 없는 문자가 있는 경우, 해당 축의 값을 모두 더하여 출력 배열에 나타나지 않도록 합니다.

출력 축은 어떤 순서로든 반환될 수 있습니다.
몇 가지 예시를 살펴보면 훨씬 더 명확해집니다.
```
# Transposes
np.einsum("ij->ji")
# matmul
np.einsum("ij,jk->ik")
# matmuls a list of matrices against a single matrix
np.einsum("hij,jk->hik")
# Dot-product
np.einsum("i,i->")
# Element-wise multiplication
np.einsum("ijk,ijk->ijk")
# Element-wise multiplies and sums everything.
np.einsum("ijk,ijk->")
```
Keras에서는 두 가지 방법으로 einsum을 사용할 수 있습니다. keras.ops.einsum은 np.einsum을 대체하는 함수이고, keras.layers.EinsumDense는 matmul 연산 대신 einsum 연산을 사용하는 Dense 레이어입니다.

#### Dot-product attention

번역 RNN을 다시 살펴보고 선택적 어텐션 개념을 추가해 보겠습니다. 단일 토큰을 예측하는 경우를 생각해 봅시다. 소스 및 타겟 시퀀스를 GRU 레이어에 통과시키면 예측하려는 타겟 토큰을 나타내는 벡터와 소스 텍스트의 각 단어를 나타내는 벡터 시퀀스를 얻게 됩니다.

어텐션을 통해 모델이 현재 예측하려는 단어와의 관련성을 기준으로 소스 시퀀스의 모든 벡터에 점수를 매길 수 있도록 하는 것이 목표입니다(그림 15.4). 소스 토큰의 벡터 표현이 높은 점수를 받으면 특히 중요하다고 간주하고, 그렇지 않으면 덜 중요하게 여깁니다. 지금은 score(target_vector, source_vector)라는 함수가 있다고 가정해 보겠습니다.

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch15/attention.6007731a.png" width="600"><br>Figure 15.4: Attention assigns a relevance score to each vector in a source for each vector in a target sequence.</p>

어텐션 메커니즘이 제대로 작동하려면 중요한 토큰에 대한 정보를 소스 및 타겟 시퀀스의 총 길이만큼 길어질 수 있는 루프를 통해 전달하는 것을 피해야 합니다. 바로 이 지점에서 RNN이 한계를 드러내기 시작합니다. 이를 해결하는 간단한 방법은 계산된 점수를 기반으로 모든 소스 벡터의 가중 합을 구하는 것입니다. 또한 특정 타겟에 대한 모든 어텐션 점수의 합이 1이면 가중 합이 예측 가능한 크기를 가지므로 편리합니다. 이를 위해 점수에 소프트맥스 함수를 적용할 수 있습니다. NumPy 의사 코드로는 다음과 같습니다.
```
scores = [score(target, source) for source in sources]
scores = softmax(scores)
combined = np.sum(scores * sources)
```
하지만 이 관련성 점수는 어떻게 계산해야 할까요? 연구자들이 처음 어텐션 메커니즘을 다룰 때, 이 질문은 중요한 연구 주제였습니다. 가장 간단한 접근 방식 중 하나가 가장 효과적이라는 것이 밝혀졌습니다. 타겟 벡터와 소스 벡터 사이의 거리를 간단하게 측정하기 위해 내적을 사용할 수 있습니다. 소스 벡터와 타겟 벡터가 서로 가까우면, 소스 토큰이 예측과 관련성이 높다고 가정합니다. 이 장의 마지막 부분에서 이러한 가정이 직관적으로 타당한 이유를 살펴보겠습니다.

이제 의사 코드를 업데이트해 보겠습니다. 전체 타겟 시퀀스를 한 번에 처리하도록 코드를 더 완벽하게 만들 수 있습니다. 이는 이전 코드를 타겟 시퀀스의 각 토큰에 대해 반복문으로 실행하는 것과 같습니다. 타겟과 소스 모두 시퀀스인 경우, 어텐션 점수는 행렬로 표현됩니다. 각 행은 가중합에서 타겟 단어가 소스 단어에 부여하는 가치를 나타냅니다(그림 15.5 참조). 내적과 가중합을 편리하게 표현하기 위해 Einsum 표기법을 사용하겠습니다.

```
def dot_product_attention(target, source):
    # Takes the dot-product between all target and source vectors,
    # where b = batch size, t = target length, s = source length, and d
    # = vector size
    scores = np.einsum("btd,bsd->bts", target, source)
    scores = softmax(scores, axis=-1)
    # Computes a weighted sum of all source vectors for each target
    # vector
    return np.einsum("bts,bsd->btd", scores, source)

dot_product_attention(target, source)
```
<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch15/attention-scores.2932e0ff.png" width="600"><br>Figure 15.5: When both target and source are sequences, attention scores are a 2D matrix. Each row shows the attention scores for the word we are trying to predict (in green).</p>

어텐션 메커니즘의 가설 공간을 훨씬 풍부하게 만들려면 모델에 어텐션 점수를 제어하는 ​​매개변수를 제공해야 합니다. 소스 벡터와 타겟 벡터를 모두 Dense 레이어로 투영하면, 모델은 전반적인 예측 품질 향상에 도움이 되는 소스 벡터와 타겟 벡터가 가까운 최적의 공유 공간을 찾을 수 있습니다. 마찬가지로, 소스 벡터를 결합하기 전과 합산 후 완전히 별개의 공간으로 투영할 수 있도록 해야 합니다.

또한, 업계에서 표준으로 자리 잡은 입력 이름 체계를 약간 다르게 사용할 수 있습니다. 방금 작성한 코드는 대략 sum(score(target, source) * source)로 요약할 수 있습니다. 이를 입력 이름을 다르게 하여 sum(score(query, key) * value)와 같이 표현할 수도 있습니다. 이 세 개의 인자를 사용하는 버전은 더 일반적입니다. 드물지만 소스 입력의 점수를 매기는 데 사용하는 벡터와 소스 입력을 합산하는 데 사용하는 벡터가 다를 수 있기 때문입니다.

이러한 용어는 검색 엔진과 추천 시스템에서 유래했습니다. 데이터베이스에서 사진을 검색하는 검색 도구를 상상해 보세요. 여기서 "쿼리"는 검색어이고, "키"는 쿼리와 일치하는 사진 태그이며, 마지막으로 "값"은 사진 자체입니다(그림 15.6). 우리가 구축하고 있는 어텐션 메커니즘은 이러한 종류의 검색과 대략적으로 유사합니다.

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch15/query-key-value.b57cceb0.png" width="600"><br>Figure 15.6: Retrieving images from a database: the query is compared to a set of keys, and the match scores are used to rank values (images).</p>

이제 새로운 용어를 사용하여 매개변수화된 어텐션을 표현하도록 의사 코드를 업데이트해 보겠습니다.

```
query_dense = layers.Dense(dim)
key_dense = layers.Dense(dim)
value_dense = layers.Dense(dim)
output_dense = layers.Dense(dim)

def parameterized_attention(query, key, value):
    query = query_dense(query)
    key = key_dense(key)
    value = value_dense(value)
    scores = np.einsum("btd,bsd->bts", query, key)
    scores = softmax(scores, axis=-1)
    outputs = np.einsum("bts,bsd->btd", scores, value)
    return output_dense(outputs)

parameterized_attention(query=target, key=source, value=source)
```
이 블록은 완벽하게 작동하는 어텐션 메커니즘입니다! 방금 작성한 함수는 디코딩하려는 목표 단어에 따라 소스 시퀀스의 어느 위치에서든 문맥에 맞는 정보를 가져올 수 있도록 합니다.

"어텐션이 전부다"의 저자들은 시행착오를 통해 우리 메커니즘에 두 가지를 더 수정했습니다. 첫 번째는 간단한 스케일링 계수입니다. 입력 벡터가 길어지면 내적 점수가 상당히 커질 수 있는데, 이는 소프트맥스 기울기의 안정성에 영향을 미칠 수 있습니다. 해결책은 간단합니다. 소프트맥스 점수를 약간 줄이면 됩니다. 벡터 길이의 제곱근으로 스케일링하면 어떤 벡터 크기에도 잘 작동합니다.

두 번째는 어텐션 메커니즘의 표현력과 관련이 있습니다. 우리가 사용하는 소프트맥스 합은 강력합니다. 시퀀스의 멀리 떨어진 부분들을 직접 연결할 수 있게 해주기 때문입니다. 하지만 이 합산 방식은 다소 투박합니다. 모델이 한 번에 너무 많은 토큰에 주의를 기울이려고 하면 개별 소스 토큰의 흥미로운 특징들이 결합된 표현에서 "희석"될 수 있습니다. 효과적인 간단한 방법은 동일한 시퀀스에 대해 여러 개의 서로 다른 어텐션 헤드를 사용하여 서로 다른 매개변수로 동일한 계산을 수행함으로써 이 어텐션 연산을 여러 번 수행하는 것입니다.

```
query_dense = [layers.Dense(head_dim) for i in range(num_heads)]
key_dense = [layers.Dense(head_dim) for i in range(num_heads)]
value_dense = [layers.Dense(head_dim) for i in range(num_heads)]
output_dense = layers.Dense(head_dim * num_heads)

def multi_head_attention(query, key, value):
    head_outputs = []
    for i in range(num_heads):
        query = query_dense[i](query)
        key = key_dense[i](key)
        value = value_dense[i](value)
        scores = np.einsum("btd,bsd->bts", target, source)
        scores = softmax(scores / math.sqrt(head_dim), axis=-1)
        head_output = np.einsum("bts,bsd->btd", scores, source)
        head_outputs.append(head_output)
    outputs = ops.concatenate(head_outputs, axis=-1)
    return output_dense(outputs)

multi_head_attention(query=target, key=source, value=source)
```
쿼리와 키를 서로 다르게 투영함으로써, 하나의 헤드는 소스 문장의 주어와 일치하도록 학습하고, 다른 헤드는 구두점에 주의를 기울일 수 있습니다. 이러한 다중 헤드 어텐션은 전체 소스 시퀀스를 단일 소프트맥스 합으로 결합해야 하는 제약을 피할 수 있습니다(그림 15.7).
<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch15/multi-head-attention.718456ad.png" width="600"><br>Figure 15.7: Multi-headed attention allows each target word to attend to different parts of the source sequence in separate partitions of the eventual output vector.</p>

물론 실제로는 이 코드를 재사용 가능한 레이어로 작성하는 것이 좋습니다. 케라스는 이러한 기능을 제공합니다. MultiHeadAttention 레이어를 사용하면 이전 코드를 다음과 같이 다시 생성할 수 있습니다.

```
multi_head_attention = keras.layers.MultiHeadAttention(
    num_heads=num_heads,
    head_dim=head_dim,
)
multi_head_attention(query=target, key=source, value=source)
```

#### Transformer encoder block

MultiHeadAttention 레이어를 사용하는 한 가지 방법은 기존 RNN 번역 모델에 추가하는 것입니다. 인코더와 디코더의 시퀀스 출력을 어텐션 레이어로 전달하고, 그 출력을 사용하여 예측 전에 목표 시퀀스를 업데이트할 수 있습니다. 어텐션은 GRU 레이어가 처리하기 어려운 텍스트 내의 장거리 의존성을 모델이 처리할 수 있도록 해줍니다. 실제로 이는 RNN 모델의 성능을 향상시키며, 2010년대 중반에 어텐션이 처음 사용된 방식입니다.

하지만 "Attention is all you need"의 저자들은 어텐션을 더 나아가 모델 내 모든 시퀀스 데이터를 처리하는 일반적인 메커니즘으로 사용할 수 있다는 점을 깨달았습니다. 지금까지는 두 시퀀스 간의 정보 전달을 처리하는 방법으로만 어텐션을 살펴보았지만, 시퀀스가 ​​자기 자신에게 어텐션하도록 하는 방법으로도 어텐션을 사용할 수 있습니다.

```
multi_head_attention(key=source, value=source, query=source)
```
이것을 셀프 어텐션(self-attention)이라고 하며, 매우 강력한 기능입니다. 셀프 어텐션을 사용하면 각 토큰이 자기 자신을 포함하여 해당 시퀀스 내의 모든 토큰에 주의를 기울일 수 있으므로, 모델은 문맥 속에서 단어를 표현하는 방법을 학습할 수 있습니다.

예를 들어 "기차가 정시에 역을 떠났다."라는 문장을 생각해 보겠습니다. 이제 문장에서 "station"이라는 단어를 생각해 보세요. 어떤 종류의 역을 말하는 걸까요? 라디오 방송국일까요? 아니면 국제 우주 정거장일까요? 셀프 어텐션을 사용하면 모델은 "station"과 "train"이라는 두 단어 쌍에 높은 주의 점수를 부여하도록 학습할 수 있으며, "train"을 표현하는 데 사용되는 벡터를 "station"이라는 단어를 표현하는 벡터에 더할 수 있습니다.

셀프 어텐션은 모델이 단어를 독립적으로 표현하는 것에서 나아가 시퀀스에 나타나는 다른 모든 토큰을 고려하여 단어를 표현하는 효과적인 방법을 제공합니다. 이는 RNN이 하는 일과 매우 유사해 보입니다. 그렇다면 RNN 레이어를 MultiHeadAttention으로 대체할 수 있을까요?

거의 그렇습니다! 하지만 완전히 그렇지는 않습니다. 모든 심층 신경망에 필수적인 요소인 비선형 활성화 함수가 여전히 필요합니다. MultiHeadAttention 레이어는 소스 시퀀스의 모든 요소에 대한 선형 투영을 결합하지만, 그게 전부입니다. 어떻게 보면, 매우 표현력이 풍부한 풀링 연산이라고 할 수 있습니다. 극단적인 경우를 생각해 보면, 토큰 길이가 1인 경우입니다. 이 경우, 어텐션 스코어 행렬은 항상 단일 행렬이 되며, 전체 레이어는 비선형성 없이 소스 시퀀스의 선형 투영으로 축소됩니다. 어텐션 레이어를 100개 쌓아도 전체 계산을 단 하나의 행렬 곱셈으로 단순화할 수 있습니다! 이것이 바로 우리 모델의 표현력에 대한 실제적인 문제입니다.

어느 시점에서 모든 순환 셀은 각 토큰에 대한 입력 벡터를 밀집 투영을 통해 전달하고 활성화 함수를 적용합니다. 우리도 이와 유사한 과정을 구현해야 합니다. "어텐션이 전부다(Attention is all you need)"의 저자들은 이를 가능한 한 가장 간단한 방법으로 다시 추가하기로 결정했습니다. 바로 두 개의 밀집 레이어 사이에 활성화 함수를 두고 피드포워드 네트워크를 쌓는 것입니다. 어텐션은 시퀀스 전체에 정보를 전달하고, 피드포워드 네트워크는 개별 시퀀스 항목의 표현을 업데이트합니다.

이제 Transformer 모델 구축을 시작할 준비가 되었습니다. 먼저 번역 모델의 인코더를 교체해 보겠습니다. 영어 단어로 이루어진 원문 시퀀스를 따라 정보를 전달하기 위해 셀프 어텐션을 사용할 것입니다. 또한 9장에서 컨볼루션 신경망을 구축할 때 특히 중요하다고 배웠던 두 가지 요소, 즉 정규화와 잔차 연결(residual connections)도 추가할 것입니다.

In [17]:
class TransformerEncoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, source, source_mask):
        residual = x = source
        mask = source_mask[:, None, :]
        x = self.self_attention(query=x, key=x, value=x, attention_mask=mask)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

여기서 사용하는 정규화 레이어는 이미지 모델에서 사용했던 배치 정규화(BatchNormalization) 레이어가 아니라는 점에 유의하세요. 배치 정규화는 시퀀스 데이터에 적합하지 않기 때문입니다. 대신, 각 시퀀스를 배치 내의 다른 시퀀스와 독립적으로 정규화하는 레이어 정규화(LayerNormalization) 레이어를 사용합니다. 다음은 NumPy 스타일의 의사 코드입니다.

```
# Input shape: (batch_size, sequence_length, embedding_dim)
def layer_normalization(batch_of_sequences):
    # To compute mean and variance, we only pool data over the last
    # axis.
    mean = np.mean(batch_of_sequences, keepdims=True, axis=-1)
    variance = np.var(batch_of_sequences, keepdims=True, axis=-1)
    return (batch_of_sequences - mean) / variance
```
배치 정규화(학습 중)와 비교해 보세요:
```
# Input shape: (batch_size, height, width, channels)
def batch_normalization(batch_of_images):
    # Pools data over the batch axis (axis 0), which creates
    # interactions between samples in a batch
    mean = np.mean(batch_of_images, keepdims=True, axis=(0, 1, 2))
    variance = np.var(batch_of_images, keepdims=True, axis=(0, 1, 2))
    return (batch_of_images - mean) / variance
```
배치 정규화(BatchNormalization)는 여러 샘플의 정보를 수집하여 특징 평균과 분산에 대한 정확한 통계를 얻는 반면, 레이어 정규화(LayerNormalization)는 각 시퀀스 내의 데이터를 개별적으로 풀링하므로 시퀀스 데이터에 더 적합합니다.

또한, 멀티헤드 어텐션(MultiHeadAttention) 레이어에 attention_mask라는 새로운 입력을 전달합니다. 이 불리언 텐서 입력은 어텐션 스코어와 동일한 형태(batch_size, target_length, source_length)로 브로드캐스트됩니다. attention_mask가 설정되면 특정 위치의 어텐션 스코어가 0이 되어 해당 위치의 소스 토큰이 어텐션 계산에 사용되지 않게 됩니다. 이는 시퀀스 내의 어떤 토큰도 정보가 없는 패딩 토큰에 어텐션하지 않도록 하기 위함입니다. 인코더 레이어는 입력에서 패딩 토큰이 아닌 모든 토큰을 표시하는 source_mask 입력을 받아 (batch_size, 1, source_length) 형태로 업랭크하여 attention_mask로 사용합니다.

이 레이어의 입력과 출력은 동일한 형태를 가지므로 인코더 블록을 서로 쌓아 올려 입력 영어 문장을 점진적으로 더 표현력 있게 표현할 수 있다는 점에 유의하십시오.

#### Transformer decoder block

다음은 디코더 블록입니다. 이 레이어는 인코더 블록과 거의 동일하지만, 디코더가 인코더 출력 시퀀스를 입력으로 사용하도록 한다는 점이 다릅니다. 이를 위해 어텐션을 두 번 사용할 수 있습니다. 먼저 인코더처럼 셀프 어텐션 레이어를 적용하여 타겟 시퀀스의 각 위치가 다른 타겟 위치의 정보를 활용할 수 있도록 합니다. 그런 다음 소스 시퀀스와 타겟 시퀀스를 모두 입력으로 받는 멀티헤드 어텐션 레이어를 추가합니다. 이 어텐션 레이어는 인코더와 디코더 간에 정보를 전달하므로 크로스 어텐션이라고 부릅니다.

In [18]:
class TransformerDecoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.cross_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.cross_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, target, source, source_mask):
        residual = x = target
        x = self.self_attention(query=x, key=x, value=x, use_causal_mask=True)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        mask = source_mask[:, None, :]
        x = self.cross_attention(
            query=x, key=source, value=source, attention_mask=mask
        )
        x = x + residual
        x = self.cross_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

디코더 레이어는 타겟과 소스 모두를 입력으로 받습니다. TransformerEncoder와 마찬가지로, 소스 입력에서 패딩의 위치를 ​​표시하는 `source_mask`를 입력으로 받습니다(패딩이 없으면 True, 있으면 False). 이 `source_mask`는 크로스 어텐션 레이어의 `attention_mask`로 사용됩니다.

디코더의 셀프 어텐션 레이어에는 다른 유형의 어텐션 마스크가 필요합니다. RNN 디코더를 구축할 때 양방향 RNN을 사용하지 않았던 것을 기억하세요. 양방향 RNN을 사용했다면 모델이 예측하려는 레이블을 특징으로 인식하여 속임수를 쓸 수 있었기 때문입니다! 어텐션은 본질적으로 양방향입니다. 셀프 어텐션에서는 타겟 시퀀스의 어떤 토큰 위치든 다른 어떤 위치에도 어텐션을 적용할 수 있습니다. 특별한 조치를 취하지 않으면 모델은 시퀀스의 다음 토큰을 현재 레이블로 인식하고 새로운 번역을 생성하는 능력을 잃게 됩니다.

이러한 문제를 해결하기 위해 특별한 "인과적" 어텐션 마스크를 사용하여 단방향 정보 흐름을 구현할 수 있습니다. 예를 들어, 다음과 같이 하삼각 영역에 1이 포함된 어텐션 마스크를 입력으로 받는다고 가정해 보겠습니다.
```
[
    [1, 0, 0, 0, 0],
    [1, 1, 0, 0, 0],
    [1, 1, 1, 0, 0],
    [1, 1, 1, 1, 0],
    [1, 1, 1, 1, 1],
]
```
각 행 i는 위치 i에 있는 대상 토큰에 대한 어텐션 마스크로 해석될 수 있습니다. 첫 번째 행에서 첫 번째 토큰은 자신에게만 어텐션할 수 있습니다. 두 번째 행에서 두 번째 토큰은 첫 번째와 두 번째 토큰 모두에 어텐션할 수 있으며, 이러한 방식으로 계속됩니다. 이는 정보가 시퀀스에서 앞으로만 전파되고 뒤로는 전파되지 않는 RNN 레이어와 동일한 효과를 제공합니다. Keras에서는 MultiHeadAttention 레이어를 호출할 때 use_casual_mask를 전달하여 이 하삼각 마스크를 지정할 수 있습니다. 그림 15.8은 Transformer 모델로 구성될 때 인코더 및 디코더 레이어의 시각적 표현을 보여줍니다.

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch15/encoder-decoder.d979dbbc.png" width="600"><br>Figure 15.8: A visual representation of the computations for both TransformerEncoder and TransformerDecoder blocks</p>

#### Sequence-to-sequence learning with a Transformer

이 모든 것을 종합해 보겠습니다. RNN 모델과 동일한 기본 설정을 사용하되, GRU 레이어를 TransformerEncoder와 TransformerDecoder로 대체합니다. 피드포워드 블록을 제외한 모델 전체에서 임베딩 크기는 256으로 유지합니다. 피드포워드 블록에서는 비선형 연산 전에 임베딩 크기를 2048로 확대하고, 비선형 연산 후에는 다시 모델의 은닉층 크기로 축소합니다. 이처럼 중간 차원을 크게 설정하는 것이 실제 구현에 효과적입니다.

In [19]:
hidden_dim = 256
intermediate_dim = 2048
num_heads = 8

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,
)

target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = layers.Embedding(vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions)

트랜스포머 모델의 요약을 살펴보겠습니다.

In [20]:
transformer.summary(line_length=80)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ english (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding (Embedding) │ (None, None, 256) │   3,840,000 │ english[0][0]      │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal (NotEqual)  │ (None, None)      │           0 │ english[0][0]      │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ spanish (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_encoder   │ (None, None, 256) │   1,315,072 │ embedding[0][0],   │
│ (TransformerEncoder)  │                   │             │ not_equal[0][0]    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal_1           │ (None, None)      │           0 │ english[0][0]      │
│ (NotEqual)            │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_1           │ (None, None, 256) │   3,840,000 │ spanish[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_decoder   │ (None, None, 256) │   1,578,752 │ transformer_encod… │
│ (TransformerDecoder)  │                   │             │ not_equal_1[0][0], │
│                       │                   │             │ embedding_1[0][0]  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout_3 (Dropout)   │ (None, None, 256) │           0 │ transformer_decod… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dense_4 (Dense)       │ (None, None,      │   3,855,000 │ dropout_3[0][0]    │
│                       │ 15000)            │             │                    │
└───────────────────────┴───────────────────┴─────────────┴────────────────────┘

 Total params: 14,428,824 (55.04 MB)

 Trainable params: 14,428,824 (55.04 MB)

 Non-trainable params: 0 (0.00 B)

저희 모델은 이전에 학습시킨 GRU 번역 모델과 거의 동일한 구조를 가지고 있으며, 순환 레이어 대신 어텐션 메커니즘을 사용하여 시퀀스 전체에 정보를 전달합니다. 이제 모델을 학습시켜 보겠습니다.

In [21]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
transformer.fit(train_ds, epochs=15, validation_data=val_ds)

Epoch 1/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 708s 541ms/step - accuracy: 0.3733 - loss: 1.4606 - val_accuracy: 0.5018 - val_loss: 1.0135
Epoch 2/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 783s 601ms/step - accuracy: 0.5413 - loss: 0.9488 - val_accuracy: 0.5755 - val_loss: 0.8200
Epoch 3/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 766s 589ms/step - accuracy: 0.6093 - loss: 0.7450 - val_accuracy: 0.6021 - val_loss: 0.7570
Epoch 4/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 735s 564ms/step - accuracy: 0.6516 - loss: 0.6234 - val_accuracy: 0.6164 - val_loss: 0.7279
Epoch 5/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 764s 587ms/step - accuracy: 0.6847 - loss: 0.5387 - val_accuracy: 0.6236 - val_loss: 0.7236
Epoch 6/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 776s 596ms/step - accuracy: 0.7108 - loss: 0.4755 - val_accuracy: 0.6311 - val_loss: 0.7237
Epoch 7/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 768s 590ms/step - accuracy: 0.7325 - loss: 0.4274 - val_accuracy: 0.6328 - val_loss: 0.7288
Epoch 8/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 751s 577ms/step - ac

학습 후 정확도는 약 58%에 도달했습니다. 즉, 모델이 스페인어 문장에서 다음 단어를 평균 58%의 확률로 정확하게 예측한다는 뜻입니다. 뭔가 이상합니다. 학습 결과가 RNN 모델보다 7%포인트나 떨어집니다. 이 Transformer 아키텍처가 과대광고된 만큼 강력하지 않거나, 구현 과정에서 무언가 잘못된 부분이 있는 것 같습니다. 무엇이 문제인지 찾을 수 있나요?

이 부분은 표면적으로는 시퀀스 모델에 대한 내용입니다. 이전 장에서 단어 순서가 의미 전달에 얼마나 중요한지 살펴보았습니다. 하지만 우리가 방금 구축한 Transformer는 사실 시퀀스 모델이 아닙니다. 눈치채셨나요? 이 모델은 시퀀스 토큰을 서로 독립적으로 처리하는 완전 연결 레이어와 토큰들을 하나의 집합으로 인식하는 어텐션 레이어로 구성되어 있습니다. 시퀀스에서 토큰의 순서를 바꾸더라도 쌍별 어텐션 점수와 문맥 인식 표현은 동일하게 유지됩니다. 모든 영어 원문 문장의 모든 단어 순서를 완전히 바꿔도 모델은 이를 알아채지 못하고 동일한 정확도를 유지합니다. 어텐션은 시퀀스 요소 쌍 간의 관계에 초점을 맞춘 집합 처리 메커니즘입니다. 즉, 이러한 요소가 시퀀스의 시작, 끝 또는 중간에 나타나는지 여부는 고려하지 않습니다. 그렇다면 왜 트랜스포머를 시퀀스 모델이라고 부를까요? 그리고 단어 순서를 고려하지 않는 트랜스포머가 어떻게 기계 번역에 적합할 수 있을까요?

RNN의 경우, 레이어의 연산이 순서를 인식하도록 설계되었습니다. 하지만 트랜스포머의 경우, 임베딩된 시퀀스 자체에 위치 정보를 직접 삽입합니다. 이를 위치 임베딩이라고 합니다. 자세히 살펴보겠습니다.

#### Embedding positional information

위치 임베딩의 기본 아이디어는 매우 간단합니다. 모델이 단어 순서 정보를 활용할 수 있도록 각 단어 임베딩에 문장 내 단어의 위치를 ​​추가하는 것입니다. 입력 단어 임베딩은 두 가지 구성 요소로 이루어져 있습니다. 하나는 특정 문맥과 관계없이 단어를 나타내는 일반적인 단어 벡터이고, 다른 하나는 현재 문장에서 단어의 위치를 ​​나타내는 위치 벡터입니다. 모델은 이 추가 정보를 어떻게 가장 효과적으로 활용할지 스스로 판단할 것입니다.

위치 정보를 추가하는 가장 간단한 방법은 각 단어의 위치를 ​​임베딩 벡터에 연결하는 것입니다. 벡터에 "위치" 축을 추가하고, 순서상 첫 번째 단어는 0, 두 번째 단어는 1 등으로 값을 채우면 됩니다.

하지만 이 방법은 위치 값이 매우 큰 정수일 수 있기 때문에 이상적이지 않을 수 있습니다. 이는 임베딩 벡터의 값 범위를 왜곡할 수 있습니다. 아시다시피 신경망은 매우 큰 입력값이나 불연속적인 입력 분포를 좋아하지 않습니다.

"Attention is all you need"의 저자들은 단어 위치를 인코딩하기 위해 흥미로운 기법을 사용했습니다. 단어 임베딩에 위치에 따라 주기적으로 변하는 [-1, 1] 범위의 값을 가진 벡터를 추가한 것입니다(이를 위해 코사인 함수를 사용했습니다). 이 기법은 작은 값으로 이루어진 벡터를 통해 넓은 범위의 모든 정수를 고유하게 특징화할 수 있는 방법을 제공합니다. 기발한 방법이지만, 더 간단하고 효과적인 방법이 있습니다. 단어 인덱스를 임베딩하는 것과 같은 방식으로 위치 임베딩 벡터를 학습하는 것입니다. 그런 다음 위치 임베딩을 해당 단어 임베딩에 추가하여 위치를 인식하는 단어 임베딩을 얻습니다. 이를 위치 임베딩이라고 합니다. 이제 구현해 보겠습니다.

In [22]:
from keras import ops

class PositionalEmbedding(keras.Layer):
    def __init__(self, sequence_length, input_dim, output_dim):
        super().__init__()
        self.token_embeddings = layers.Embedding(input_dim, output_dim)
        self.position_embeddings = layers.Embedding(sequence_length, output_dim)

    def call(self, inputs):
        positions = ops.cumsum(ops.ones_like(inputs), axis=-1) - 1
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

우리는 이 PositionalEmbedding 레이어를 일반적인 Embedding 레이어처럼 사용할 것입니다. 이제 Transformer를 두 번째로 학습시키면서 실제로 어떻게 작동하는지 살펴보겠습니다.

In [23]:
hidden_dim = 256
intermediate_dim = 2056
num_heads = 8

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,
)

target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions)

위치 임베딩이 모델에 추가되었으니 다시 학습을 시도해 보겠습니다.

In [24]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
transformer.fit(train_ds, epochs=30, validation_data=val_ds)

Epoch 1/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 793s 606ms/step - accuracy: 0.3895 - loss: 1.4218 - val_accuracy: 0.5333 - val_loss: 0.9518
Epoch 2/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 745s 572ms/step - accuracy: 0.5756 - loss: 0.8869 - val_accuracy: 0.6172 - val_loss: 0.7480
Epoch 3/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 773s 593ms/step - accuracy: 0.6460 - loss: 0.6889 - val_accuracy: 0.6490 - val_loss: 0.6786
Epoch 4/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 772s 593ms/step - accuracy: 0.6855 - loss: 0.5764 - val_accuracy: 0.6635 - val_loss: 0.6430
Epoch 5/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 732s 563ms/step - accuracy: 0.7136 - loss: 0.5006 - val_accuracy: 0.6737 - val_loss: 0.6260
Epoch 6/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 772s 593ms/step - accuracy: 0.7357 - loss: 0.4432 - val_accuracy: 0.6785 - val_loss: 0.6246
Epoch 7/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 772s 593ms/step - accuracy: 0.7560 - loss: 0.3978 - val_accuracy: 0.6803 - val_loss: 0.6279
Epoch 8/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 733s 563ms/step - ac

위치 정보를 모델에 다시 도입하니 결과가 훨씬 좋아졌습니다. 다음 단어를 예측하는 데 67%의 정확도를 달성했습니다. 이는 GRU 모델에 비해 눈에 띄게 향상된 수치이며, 특히 이 모델이 GRU 모델의 절반에 불과한 파라미터를 사용한다는 점을 고려하면 더욱 인상적입니다.

이번 학습 과정에서 또 하나 중요한 점이 있습니다. 학습 속도가 RNN보다 훨씬 빠르다는 것입니다. 각 에포크에 걸리는 시간이 약 3분의 1로 단축되었습니다. 파라미터 개수를 RNN 모델과 동일하게 하더라도 마찬가지일 것이며, 이는 GRU 레이어의 반복적인 상태 전달을 제거한 덕분입니다. 어텐션 메커니즘을 사용하면 학습 중에 반복적인 연산을 처리할 필요가 없으므로 GPU나 TPU에서 전체 어텐션 연산을 한 번에 처리할 수 있습니다. 따라서 Transformer는 가속기에서 더 빠른 학습 속도를 제공합니다.

이제 새로 학습된 Transformer를 사용하여 생성 연산을 다시 실행해 보겠습니다. RNN 샘플링에 사용했던 것과 동일한 코드를 사용하면 됩니다.

In [25]:
import numpy as np

spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    for i in range(sequence_length):
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        tokenized_target_sentence = tokenized_target_sentence[:, :-1]
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = transformer.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

-
He never travels without taking an alarm clock with him.
[start] Él nunca cantado auriculares sin hacerle una alarma a él [end]
-
I'm sending you a book.
[start] te voy a enviar un libro [end]
-
I studied it thoroughly.
[start] lo estudié con un daño [end]
-
When I saw him last, he was still a child.
[start] cuando le vi [UNK] él todavía era niño [end]
-
We're all in this together.
[start] estamos todos juntos [end]


생성 코드를 실행하면 다음과 같은 출력이 나타납니다.

주관적으로 볼 때, 트랜스포머는 GRU 기반 번역 모델보다 훨씬 뛰어난 성능을 보여줍니다. 여전히 장난감 모델에 가깝지만, 훨씬 더 나은 장난감 모델입니다.

트랜스포머는 텍스트 처리 모델에 대한 관심이 폭발적으로 증가한 기반을 마련한 강력한 아키텍처입니다. 딥러닝 모델 중에서도 상당히 복잡한 구조를 가지고 있습니다. 이러한 구현 세부 사항들을 모두 살펴본 후에는, 모든 것이 다소 임의적인 것처럼 보일 수 있다는 반론이 나올 수도 있습니다. 너무나 많은 세부 사항들을 그대로 받아들여야 하는데, 이러한 레이어 선택과 구성이 최적이라는 것을 어떻게 확신할 수 있을까요?

답은 간단합니다. 최적이 아닙니다. 수년에 걸쳐 어텐션, 정규화, 위치 임베딩 등을 변경하여 트랜스포머 아키텍처를 개선하는 여러 가지 방안이 제시되었습니다. 오늘날 많은 새로운 모델들은 시퀀스 길이가 매우 길어짐에 따라 어텐션을 계산 복잡성이 낮은 다른 방식으로 대체하고 있습니다. 결국, 아마도 이 책을 읽을 때쯤에는 언어 모델링에 사용되는 주요 아키텍처로서 트랜스포머를 대체하는 무언가가 등장했을지도 모릅니다.

트랜스포머로부터 배울 수 있는 것들은 앞으로도 오랫동안 가치를 지닐 것입니다. 이 장의 마지막 부분에서는 트랜스포머가 왜 그렇게 효과적인지 논의할 것입니다. 하지만 머신러닝 분야 전체가 경험적 검증을 통해 발전해 왔다는 점을 기억할 필요가 있습니다. 어텐션 모델은 순환신경망(RNN)을 강화하려는 시도에서 탄생했고, 수많은 사람들이 수년간 시행착오를 거친 끝에 트랜스포머가 개발되었습니다. 이러한 과정이 아직 끝나지 않았다고 생각할 만한 이유는 거의 없습니다.

### Classification with a pretrained Transformer

"Attention is all you need"라는 글이 나온 후, 사람들은 Transformer 학습의 확장성이 얼마나 뛰어난지, 특히 기존 모델들과 비교했을 때 얼마나 큰 장점인지 주목하기 시작했습니다. 앞서 언급했듯이, Transformer의 가장 큰 장점 중 하나는 RNN보다 학습 속도가 빠르다는 점입니다. GPU나 TPU를 사용할 때 학습 과정에서 반복문이 필요 없다는 점은 매우 유리합니다.

또한 Transformer는 데이터 요구량이 매우 높은 모델 아키텍처입니다. 지난 섹션에서 이를 직접 경험해 보았습니다. RNN 번역 모델은 5 에포크 정도 학습 후 검증 성능이 정체된 반면, Transformer 모델은 30 에포크 학습 후에도 검증 점수가 계속 향상되었습니다.

이러한 관찰을 바탕으로 많은 사람들이 더 많은 데이터, 레이어, 파라미터를 사용하여 Transformer를 확장하는 시도를 했고, 그 결과는 매우 좋았습니다. 이는 학습 비용이 수백만 달러에 달하지만 텍스트 도메인의 다양한 문제에서 훨씬 뛰어난 성능을 보이는 대규모 사전 학습 모델에 대한 수요 증가로 이어졌습니다.

텍스트 섹션의 마지막 코드 예제에서는 사전 학습된 Transformer 모델을 사용하여 IMDb 텍스트 분류 문제를 다시 살펴보겠습니다.

#### Pretraining a Transformer encoder

자연어 처리(NLP) 분야에서 인기를 얻은 최초의 사전 학습된 트랜스포머 중 하나는 BERT(Bidirectional Encoder Representations from Transformers)[2]였습니다. 이 논문과 모델은 "Attention Is All You Need"가 발표된 지 1년 후에 공개되었습니다. 모델 구조는 방금 구축한 번역 트랜스포머의 인코더 부분과 정확히 동일합니다. 이 인코더 모델은 양방향성을 가지는데, 시퀀스의 모든 위치가 앞뒤 위치에 어텐션을 적용할 수 있다는 의미입니다. 즉, 입력 텍스트의 풍부한 표현을 계산하는 데는 적합하지만, 반복적인 생성 작업을 수행하도록 설계된 모델은 아닙니다.

BERT는 1억에서 3억 개의 파라미터를 가진 크기로 학습되었는데, 이는 방금 학습한 1400만 개의 파라미터를 가진 트랜스포머보다 훨씬 큽니다. 따라서 모델이 좋은 성능을 내려면 많은 양의 학습 데이터가 필요했습니다. 이를 위해 저자들은 마스크 언어 모델링(Masked Language Modeling)이라는 고전적인 언어 모델링 방식을 변형하여 사용했습니다. 모델을 사전 학습하기 위해 텍스트 시퀀스에서 약 15%의 토큰을 특수한 [MASK] 토큰으로 대체했습니다. 이 모델은 학습 과정에서 원래 마스킹된 토큰 값을 예측하려고 시도합니다. 고전적인 언어 모델(때때로 인과 언어 모델이라고도 함)이 p(토큰|이전 토큰)을 예측하려고 하는 반면, 마스크 언어 모델은 p(토큰|주변 토큰)을 예측하려고 합니다.

이 학습 방식은 비지도 학습입니다. 입력 텍스트에 대한 레이블이 필요하지 않습니다. 어떤 텍스트 시퀀스에서든 임의의 토큰을 선택하여 마스킹할 수 있습니다. 덕분에 저자들은 이 규모의 모델을 학습하는 데 필요한 방대한 양의 텍스트 데이터를 쉽게 확보할 수 있었습니다. 대부분의 데이터는 위키피디아에서 가져왔습니다.

BERT가 출시되었을 당시 사전 학습된 단어 임베딩을 사용하는 것은 이미 일반적인 관행이었습니다. 지난 장에서 이를 직접 살펴보았습니다. 하지만 전체 트랜스포머를 사전 학습함으로써 훨씬 강력한 기능을 구현할 수 있었습니다. 바로 주변 단어들의 맥락 속에서 단어 임베딩을 계산할 수 있게 된 것입니다. 트랜스포머 덕분에 당시에는 상상할 수 없었던 규모와 품질로 이러한 작업을 수행할 수 있었습니다.

BERT 개발자들은 방대한 양의 텍스트로 사전 학습된 이 모델을 활용하여 당시 여러 자연어 처리 벤치마크에서 최첨단 성능을 달성했습니다. 이는 대규모 사전 학습 모델을 사용하고 미세 조정은 최소한으로만 수행하는 방향으로 자연어 처리 분야에 뚜렷한 변화를 가져왔습니다. 이제 직접 시도해 보겠습니다.

#### Loading a pretrained Transformer

여기서는 BERT 대신 Robustly Optimized BERT의 약자인 RoBERTa[3]라는 후속 모델을 사용해 보겠습니다. RoBERTa는 BERT 아키텍처를 약간 단순화했지만, 가장 중요한 것은 성능 향상을 위해 더 많은 훈련 데이터를 사용했다는 점입니다. BERT는 주로 위키피디아에서 가져온 16GB의 영어 텍스트를 사용했습니다. RoBERTa 개발자들은 웹 전체에서 160GB의 텍스트를 사용했습니다. 당시 RoBERTa를 훈련하는 데 수십만 달러가 소요된 것으로 추정됩니다. 이러한 추가 훈련 데이터 덕분에 동일한 전체 매개변수 개수에서 모델 성능이 눈에 띄게 향상되었습니다.

사전 훈련된 모델을 사용하려면 몇 가지가 필요합니다.

* 일치하는 토크나이저 - 사전 훈련된 모델 자체와 함께 사용됩니다. 모든 텍스트는 사전 훈련 시와 동일한 방식으로 토큰화되어야 합니다. IMDb 리뷰의 단어가 사전 훈련 시와 다른 토큰 인덱스에 매핑되는 경우, 모델에서 각 토큰의 학습된 표현을 사용할 수 없습니다.
* 일치하는 모델 아키텍처 — 사전 학습된 모델을 사용하려면 사전 학습에 사용된 내부 연산을 정확하게 재현해야 합니다.
* 사전 학습된 가중치 — 이 가중치는 1,024개의 GPU와 수십억 개의 입력 단어를 사용하여 약 하루 동안 모델을 학습시켜 생성되었습니다.

토크나이저와 아키텍처 코드를 직접 재현하는 것은 그리 어렵지 않습니다. 모델 내부 구조는 이전에 구축했던 TransformerEncoder와 거의 정확히 일치합니다. 그러나 모델 구현을 일치시키는 것은 시간이 많이 소요되는 작업이므로, 이 책의 앞부분에서처럼 KerasHub 라이브러리를 사용하여 Keras용 사전 학습된 모델 구현에 접근할 수 있습니다.

KerasHub를 사용하여 RoBERTa 토크나이저와 모델을 로드해 보겠습니다. 특수 생성자 from_preset()을 사용하여 사전 학습된 모델의 가중치, 구성 및 토크나이저 자산을 디스크에서 로드할 수 있습니다. RoBERTa 논문과 함께 공개된 몇 가지 사전 학습된 체크포인트 중 가장 작은 RoBERTa 기본 모델을 로드하겠습니다.


In [26]:
import keras_hub

tokenizer = keras_hub.models.Tokenizer.from_preset("roberta_base_en")
backbone = keras_hub.models.Backbone.from_preset("roberta_base_en")

100%|██████████████████████████████████████████████████████████████████████████████████| 445/445 [00:00<00:00, 589kB/s]


100%|██████████████████████████████████████████████████████████████████████████████████| 686/686 [00:00<00:00, 657kB/s]


TypeError: <class 'keras_hub.src.models.roberta.roberta_tokenizer.RobertaTokenizer'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras_hub.src.models.roberta.roberta_tokenizer', 'class_name': 'RobertaTokenizer', 'config': {'name': 'roberta_tokenizer', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'int32'}, 'registered_name': None}, 'config_file': 'tokenizer.json', 'sequence_length': None, 'add_prefix_space': False, 'unsplittable_tokens': ['<s>', '<pad>', '</s>', '<mask>']}, 'registered_name': 'keras_hub>RobertaTokenizer'}.

Exception encountered: Error when deserializing class 'RobertaTokenizer' using config={'name': 'roberta_tokenizer', 'trainable': True, 'dtype': 'int32', 'config_file': 'tokenizer.json', 'sequence_length': None, 'add_prefix_space

토크나이저는 예상대로 텍스트를 정수 시퀀스로 변환합니다. 지난 장에서 만들었던 SubWordTokenizer를 기억하시나요? RoBERTa의 토크나이저는 모든 언어의 유니코드 문자를 처리할 수 있도록 약간 수정된 것을 제외하면 거의 동일합니다.

RoBERTa의 사전 학습 데이터셋 규모를 고려할 때, 서브워드 토큰화는 필수적입니다. 문자 수준 토크나이저를 사용하면 입력 시퀀스가 ​​너무 길어져 모델 학습 비용이 크게 증가합니다. 단어 수준 토크나이저를 사용하려면 웹에서 사용되는 수백만 개의 문서에 있는 모든 단어를 포괄하기 위해 방대한 어휘집이 필요합니다. 단어를 충분히 포괄하려면 어휘집 크기가 기하급수적으로 커져 트랜스포머 맨 앞의 임베딩 레이어가 감당할 수 없을 정도로 커집니다. 서브워드 토크나이저를 사용하면 모델은 단 5만 개의 단어로 구성된 어휘집으로 모든 단어를 처리할 수 있습니다.

In [ ]:
tokenizer("The quick brown fox")

방금 불러온 백본(Backbone)은 무엇일까요? 8장에서 살펴본 것처럼, 백본은 컴퓨터 비전에서 입력 이미지를 잠재 공간으로 매핑하는 네트워크를 가리키는 용어입니다. 기본적으로 예측 기능을 제외한 비전 모델이라고 할 수 있죠. KerasHub에서는 백본을 특정 작업에 특화되지 않은 사전 학습된 모델로 정의합니다. 방금 불러온 모델은 입력 시퀀스를 받아 (batch_size, sequence_length, 768) 형태의 출력 시퀀스로 임베딩하지만, 특정 손실 함수에 맞춰 설정되어 있지는 않습니다. 따라서 문장 분류, 특정 정보를 포함하는 텍스트 구간 식별, 품사 식별 등 다양한 하위 작업에 활용할 수 있습니다.

다음으로, 이 백본에 분류 기능을 추가하는 헤드를 연결하여 IMDb 리뷰 분류 미세 조정에 특화시키겠습니다. 마치 드라이버에 여러 종류의 헤드를 연결하는 것과 같습니다. 한 작업에는 십자 드라이버를, 다른 작업에는 일자 드라이버를 연결하는 것처럼 말이죠.

이제 백본을 자세히 살펴보겠습니다. 여기서는 RoBERTa의 가장 작은 버전을 사용했지만, 여전히 1억 2400만 개의 매개변수를 가지고 있으며, 이는 이 책에서 사용한 모델 중 가장 큰 규모입니다.

In [ ]:
backbone.summary(line_length=80)

RoBERTa는 12개의 Transformer 인코더 레이어를 서로 쌓아서 사용합니다. 이는 저희 번역 모델보다 훨씬 발전된 것입니다!

#### Preprocessing IMDb movie reviews

14장에서 사용했던 IMDb 로딩 코드를 변경 없이 그대로 재사용할 수 있습니다. 이 코드는 영화 리뷰 데이터를 train_dir과 test_dir에 다운로드하고, 검증 데이터셋을 val_dir로 분할합니다.

In [ ]:
import os, pathlib, shutil, random

zip_path = keras.utils.get_file(
    origin="https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz",
    fname="imdb",
    extract=True,
)

imdb_extract_dir = pathlib.Path(zip_path) / "aclImdb"
train_dir = pathlib.Path("imdb_train")
test_dir = pathlib.Path("imdb_test")
val_dir = pathlib.Path("imdb_val")

shutil.copytree(imdb_extract_dir / "test", test_dir, dirs_exist_ok=True)

val_percentage = 0.2
for category in ("neg", "pos"):
    src_dir = imdb_extract_dir / "train" / category
    src_files = os.listdir(src_dir)
    random.Random(1337).shuffle(src_files)
    num_val_samples = int(len(src_files) * val_percentage)

    os.makedirs(train_dir / category, exist_ok=True)
    os.makedirs(val_dir / category, exist_ok=True)
    for index, file in enumerate(src_files):
        if index < num_val_samples:
            shutil.copy(src_dir / file, val_dir / category / file)
        else:
            shutil.copy(src_dir / file, train_dir / category / file)

In [ ]:
from keras.utils import text_dataset_from_directory

batch_size = 16
train_ds = text_dataset_from_directory(train_dir, batch_size=batch_size)
val_ds = text_dataset_from_directory(val_dir, batch_size=batch_size)
test_ds = text_dataset_from_directory(test_dir, batch_size=batch_size)

데이터를 불러온 후, 우리는 다시 20,000개의 영화 리뷰로 구성된 학습 데이터셋과 5,000개의 영화 리뷰로 구성된 검증 데이터셋을 갖게 되었습니다.

분류 모델을 미세 조정하기 전에, 불러온 RoBERTa 토크나이저를 사용하여 영화 리뷰 데이터를 토큰화해야 합니다. 사전 학습 과정에서 RoBERTa는 번역 모델에서 했던 것과 유사하게 토큰들을 특정 형태의 "패킹" 방식으로 시퀀스로 묶었습니다. 각 시퀀스는 <s> 토큰으로 시작하고 </s> 토큰으로 끝나며, 그 뒤에는 다음과 같이 여러 개의 <pad> 토큰이 붙습니다.
```
[
    ["<s>", "the", "quick", "brown", "fox", "jumped", ".", "</s>"],
    ["<s>", "the", "panda", "slept", ".", "</s>", "<pad>", "<pad>"],
]
```
사전 학습에 사용된 토큰 순서를 최대한 일치시키는 것이 중요합니다. 이렇게 하면 모델이 더 빠르고 정확하게 학습됩니다. KerasHub는 이러한 토큰 패킹을 위한 StartEndPacker라는 레이어를 제공합니다. 이 레이어는 시작, 끝 및 패딩 토큰을 추가하고, 필요한 경우 긴 시퀀스를 지정된 시퀀스 길이로 잘라냅니다. 이제 이 레이어를 토크나이저와 함께 사용해 보겠습니다.

In [ ]:
def preprocess(text, label):
    packer = keras_hub.layers.StartEndPacker(
        sequence_length=512,
        start_value=tokenizer.start_token_id,
        end_value=tokenizer.end_token_id,
        pad_value=tokenizer.pad_token_id,
        return_padding_mask=True,
    )
    token_ids, padding_mask = packer(tokenizer(text))
    return {"token_ids": token_ids, "padding_mask": padding_mask}, label

preprocessed_train_ds = train_ds.map(preprocess)
preprocessed_val_ds = val_ds.map(preprocess)
preprocessed_test_ds = test_ds.map(preprocess)

전처리된 단일 배치를 살펴보겠습니다.

In [ ]:
next(iter(preprocessed_train_ds))

입력 데이터 전처리가 완료되었으므로 이제 모델을 미세 조정할 준비가 되었습니다.

#### Where does pretraining data come from?

트랜스포머는 데이터를 많이 필요로 하는 모델입니다. 딥러닝 역사상 전례 없는 규모의 입력 데이터를 제공할수록 성능이 향상됩니다. 최초의 트랜스포머는 수백만 개의 토큰으로 구성된 번역 데이터셋으로 학습되었습니다. 이후 수십억 개, 그리고 현재는 수조 개의 토큰으로 학습되고 있습니다. 엄청난 양의 단어들이죠.

그렇다면 이 모든 데이터는 어디에서 오는 걸까요? 답은 시간이 지남에 따라 바뀌었지만, 간단히 말하면 인터넷입니다. 또 다른 답은 이 부분이 점점 비밀이 되어가고 있다는 것입니다. 기업들은 모델 학습에 사용한 정확한 데이터셋이나 데이터 소스의 조합을 공개하지 않는 경우가 많습니다.

시대별로 사용된 몇 가지 사전 학습 데이터셋을 살펴보겠습니다.

* 최초의 트랜스포머는 400만 쌍의 문장으로 구성된 유명한 영어-독일어 번역 데이터셋으로 학습되었습니다.
* BERT는 영어 위키백과 데이터셋과 7,000권의 자가 출판 서적 데이터셋을 사용했습니다.
* ChatGPT의 전신인 GPT2는 Reddit에서 나가는 링크를 따라가며 데이터셋을 수집했습니다.
* Meta에서 출시한 사전 학습된 Transformer 모델인 Llama의 최신 버전은 "공개적으로 이용 가능한 소스에서 가져온 15조 개의 토큰 데이터"로 학습되었습니다. 데이터의 정확한 구성을 모호하게 남겨두는 것이 점점 더 일반화되고 있습니다.

다음 장에서는 사전 학습 데이터 소스의 정확한 조합이 얼마나 중요한지 살펴보겠습니다. 가능하면 모델의 데이터 출처에 항상 주의를 기울이는 것이 좋습니다. 데이터의 출처는 모델의 편향과 성능에 영향을 미치기 때문입니다.

#### Fine-tuning a pretrained Transformer

영화 리뷰 예측을 위해 백본을 미세 조정하기 전에, 먼저 이진 분류 레이블을 출력하도록 업데이트해야 합니다. 백본은 (배치 크기, 시퀀스 길이, 768) 형태의 전체 시퀀스를 출력하는데, 각 768차원 벡터는 주변 단어와의 문맥 속에서 입력 단어를 나타냅니다. 레이블을 예측하기 전에 이 시퀀스를 샘플당 하나의 벡터로 압축해야 합니다.

한 가지 방법은 전체 시퀀스에 걸쳐 평균 풀링이나 최대 풀링을 수행하여 모든 토큰 벡터의 평균을 계산하는 것입니다. 하지만 첫 번째 토큰의 표현을 풀링 값으로 사용하는 것이 조금 더 효과적입니다. 이는 모델의 어텐션 메커니즘 때문입니다. 최종 인코더 레이어의 첫 번째 위치는 시퀀스의 다른 모든 위치에 어텐션을 적용하여 정보를 가져올 수 있습니다. 따라서 평균을 내는 것과 같은 단순한 풀링 방식 대신, 어텐션을 통해 시퀀스 전체에 걸쳐 문맥적으로 정보를 풀링할 수 있습니다.

이제 백본에 분류 헤드를 추가해 보겠습니다. 또한 출력 예측을 생성하기 전에 비선형성을 포함하는 최종 밀집 투영을 하나 더 추가할 것입니다.

In [ ]:
inputs = backbone.input
x = backbone(inputs)
x = x[:, 0, :]
x = layers.Dropout(0.1)(x)
x = layers.Dense(768, activation="relu")(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
classifier = keras.Model(inputs, outputs)

이제 IMDb 데이터셋을 사용하여 모델을 미세 조정하고 평가할 준비가 되었습니다.

In [ ]:
classifier.compile(
    optimizer=keras.optimizers.Adam(5e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
classifier.fit(
    preprocessed_train_ds,
    validation_data=preprocessed_val_ds,
)

마지막으로 학습된 모델을 평가해 보겠습니다.

In [ ]:
classifier.evaluate(preprocessed_test_ds)

단 한 번의 학습 에포크 만에 우리 모델은 93%의 정확도를 달성했는데, 이는 지난 장에서 기록했던 90%에서 눈에 띄게 향상된 수치입니다. 물론, 이 모델은 이전에 구축했던 간단한 바이그램 분류기보다 훨씬 더 많은 비용이 들지만, 이렇게 큰 모델을 사용하는 데에는 분명한 이점이 있습니다. 게다가 이 모든 성과는 비교적 작은 규모의 RoBERTa 모델로 달성한 것입니다. 3억 개의 파라미터를 가진 더 큰 모델을 사용한다면 95% 이상의 정확도를 얻을 수 있을 것입니다.

### What makes the Transformer effective?

2013년 구글에서 토마스 미콜로프와 그의 동료들은 놀라운 사실을 발견했습니다. 그들은 지난 장에서 다룬 CBOW(Continuous Bag of Words) 임베딩과 유사한 "Word2Vec"이라는 사전 학습된 임베딩을 구축하고 있었습니다. CBOW 모델과 마찬가지로, 그들의 학습 목표는 단어 간의 상관관계를 임베딩 공간에서의 거리 관계로 변환하는 것이었습니다. 즉, 어휘의 각 단어에 벡터를 연결하고, 자주 함께 나타나는 단어를 나타내는 벡터 간의 내적(코사인 근접도)이 1에 가깝고, 드물게 함께 나타나는 단어를 나타내는 벡터 간의 내적이 0에 가깝도록 최적화했습니다.

그들은 결과적으로 얻어진 임베딩 공간이 의미적 유사성뿐만 아니라 훨씬 더 많은 것을 포착한다는 것을 발견했습니다. 그것은 일종의 "단어 연산"과 같은, 새로운 형태의 학습을 특징으로 했습니다. 공간에는 여러 남성 명사에 더해 여성 명사에 가장 가까운 값을 얻을 수 있는 벡터가 존재했습니다. 예를 들어 V(king) - V(man) + V(woman) = V(queen)과 같은 성별 벡터였습니다. 이는 매우 놀라운 발견이었습니다. 모델이 명시적으로 이러한 기능을 학습하도록 설계된 적이 없었기 때문입니다. 복수형 벡터, 야생 동물 이름에서 가장 유사한 애완동물 이름을 찾는 벡터 등 이러한 마법의 벡터가 수십 개나 있는 듯했습니다.

약 10년 후, 우리는 대규모 사전 학습된 Transformer 모델의 시대에 살고 있습니다. 겉으로 보기에는 이러한 모델들이 초기 Word2Vec 모델과는 완전히 다른 것처럼 보입니다. Transformer는 완벽하게 유창한 언어를 생성할 수 있는데, 이는 Word2Vec이 전혀 불가능했던 일입니다. 다음 장에서 살펴보겠지만, 이러한 모델은 거의 모든 주제에 대해 해박한 지식을 가진 것처럼 보일 수 있습니다. 하지만 사실 Transformer는 Word2Vec과 많은 공통점을 가지고 있습니다.

두 모델 모두 토큰(단어 또는 하위 단어)을 벡터 공간에 임베딩하려고 합니다. 두 모델 모두 임베딩 공간을 학습하는 데 있어 동일한 기본 원리를 사용합니다. 즉, 함께 나타나는 토큰은 임베딩 공간에서 가까운 위치에 있게 됩니다. 토큰을 비교하는 데 사용되는 거리 함수는 두 경우 모두 코사인 거리입니다. 임베딩 공간의 차원조차도 유사합니다. 각 단어를 표현하기 위해 1,000에서 10,000 사이의 차원을 가진 벡터를 사용합니다.

여기서 다음과 같은 의문이 들 수 있습니다. 트랜스포머는 시퀀스에서 누락된 단어를 예측하도록 훈련된 것이지, 임베딩 공간에서 토큰을 그룹화하도록 훈련된 것이 아닙니다. 언어 모델의 손실 함수가 Word2Vec의 목표인 함께 나타나는 토큰 간의 내적 최대화와 어떤 관련이 있을까요? 그 답은 어텐션 메커니즘에 있습니다.

어텐션은 트랜스포머 아키텍처에서 가장 중요한 구성 요소입니다. 어텐션은 이전 임베딩 공간에서 토큰 임베딩을 선형적으로 재조합하여 새로운 토큰 임베딩 공간을 학습하는 메커니즘입니다. 이때, 이미 서로 "가까운" 토큰(즉, 내적이 더 큰 토큰)에 더 큰 가중치를 부여합니다. 어텐션은 이미 가까운 토큰들의 벡터를 서로 끌어당기는 경향이 있으며, 시간이 지남에 따라 토큰 상관 관계가 임베딩 근접 관계(코사인 거리 기준)로 변환되는 공간을 생성합니다. 트랜스포머는 이전 임베딩 공간의 요소들을 재조합하여 점진적으로 정제된 일련의 임베딩 공간을 학습하는 방식으로 작동합니다.

어텐션은 트랜스포머에 두 가지 중요한 속성을 제공합니다.

* 트랜스포머가 학습하는 임베딩 공간은 의미적으로 연속적입니다. 즉, 임베딩 공간에서 비트만큼 이동해도 해당 토큰의 사람이 인식하는 의미는 비트만큼만 변화합니다. Word2Vec 공간 또한 이러한 속성을 보였습니다.
* 이들이 학습하는 임베딩 공간은 의미론적으로 보간적입니다. 즉, 임베딩 공간에서 두 점 사이의 중간점을 취하면 해당 토큰 사이의 "중간 의미"를 나타내는 점이 생성됩니다. 이는 각 새로운 임베딩 공간이 이전 공간의 벡터 사이를 보간하여 구축된다는 사실에서 비롯됩니다.

이는 뇌의 학습 방식과 완전히 다르지는 않습니다. 뇌의 핵심 학습 원리는 헤비안 학습, 즉 "함께 발화하는 뉴런은 함께 연결된다"는 것입니다. 신경 발화 사건(행동이나 지각 입력을 나타낼 수 있음) 간의 상관 관계는 뇌 네트워크에서 근접 관계로 변환되는데, Transformer와 Word2Vec이 상관 관계를 벡터 근접 관계로 변환하는 것과 유사합니다. 둘 다 정보 공간의 지도입니다.

물론 Word2Vec과 Transformer 사이에는 상당한 차이점이 있습니다. Word2Vec은 텍스트 생성 샘플링을 위해 설계된 것이 아닙니다. Transformer는 훨씬 더 큰 규모로 확장될 수 있으며 훨씬 더 복잡한 변환을 인코딩할 수 있습니다. 문제는 Word2Vec이 오늘날의 언어 모델과 비교했을 때, MNIST 픽셀에 대한 로지스틱 회귀 모델이 최첨단 컴퓨터 비전 모델과 비교되는 것과 같은 수준의, 말 그대로 장난감 모델이라는 점입니다. 기본적인 원리는 대부분 동일하지만, 장난감 모델은 의미 있는 표현력을 결여하고 있습니다. Word2Vec은 심지어 심층 신경망조차 아니었습니다. 얕은 단일 레이어 구조를 가지고 있었습니다. 반면, 오늘날의 Transformer 모델은 지금까지 훈련된 어떤 모델보다도 가장 강력한 표현력을 자랑합니다. 수십 개의 어텐션 및 피드포워드 레이어가 쌓여 있고, 파라미터 수는 수십억 개에 달합니다.

Word2Vec과 마찬가지로 Transformer는 토큰을 벡터 공간으로 구성하는 과정에서 유용한 의미 기능을 자연스럽게 학습합니다. 하지만 향상된 표현력과 훨씬 더 정교해진 자기회귀 최적화 목표 덕분에, 우리는 더 이상 성별 벡터나 복수형 벡터와 같은 선형 변환에만 국한되지 않습니다. 트랜스포머는 임의로 복잡한 벡터 함수를 저장할 수 있습니다. 사실, 너무 복잡해서 함수라기보다는 벡터 프로그램이라고 부르는 것이 더 정확할 정도입니다.

Word2Vec을 사용하면 복수형(고양이) → 고양이들(cats)이나 남성형(king) → 여성형(queen)과 같은 기본적인 변환을 할 수 있었습니다. 반면, 대규모 트랜스포머 모델은 셰익스피어 스타일로 시를 쓰는 마법 같은 작업, 예를 들어 새로운 시를 쓰는 마법 같은 작업도 수행할 수 있습니다. 하나의 모델에 수백만 개의 이러한 프로그램을 저장할 수도 있습니다.

트랜스포머는 데이터베이스와 유사하다고 볼 수 있습니다. 트랜스포머는 사용자가 전달하는 토큰을 통해 검색할 수 있는 정보를 저장합니다. 하지만 트랜스포머와 데이터베이스 사이에는 두 가지 중요한 차이점이 있습니다.

첫 번째 차이점은 트랜스포머가 연속적이고 보간적인 데이터베이스라는 점입니다. 데이터는 개별 항목들의 집합으로 저장되는 대신, 벡터 공간, 즉 곡선으로 저장됩니다. 이 곡선 위를 이동하며(앞서 논의했듯이 의미론적으로 연속적입니다) 인접한 관련 지점들을 탐색할 수 있습니다. 또한, 곡선 상에서 서로 다른 데이터 지점 사이를 보간하여 중간값을 찾을 수 있습니다. 이는 데이터베이스에 입력한 데이터보다 훨씬 더 많은 데이터를 검색할 수 있다는 것을 의미합니다. 물론 모든 데이터가 정확하거나 의미 있는 것은 아닙니다. 보간은 일반화를 가능하게 하지만, 잘못된 해석을 초래할 수도 있는데, 이는 오늘날 학습되는 생성형 언어 모델이 직면한 중요한 문제입니다.

두 번째 차이점은 트랜스포머가 단순히 데이터만 저장하는 것이 아니라는 점입니다. 인터넷에서 수집한 수십만 개의 문서를 기반으로 학습된 RoBERTa와 같은 모델에는 사실, 장소, 인물, 날짜, 사물, 관계 등 방대한 데이터가 저장되어 있습니다. 하지만 트랜스포머는 프로그램 데이터베이스이기도 하며, 어쩌면 프로그램의 주요 데이터베이스라고 할 수 있습니다.

이 프로그램들은 여러분이 익숙하게 다루던 프로그램과는 다릅니다. 파이썬 프로그램처럼 데이터를 단계적으로 처리하는 일련의 기호 명령으로 이루어진 것이 아닙니다. 오히려 이러한 벡터 프로그램은 잠재 임베딩 공간을 자기 자신으로 매핑하는 고도로 비선형적인 함수입니다. Word2Vec의 마법 벡터와 유사하지만 훨씬 더 복잡합니다.

다음 장에서는 트랜스포머 모델을 완전히 새로운 규모로 확장할 것입니다. 모델은 수십억 개의 매개변수를 사용하고 수조 개의 단어로 학습될 것입니다. 이러한 모델의 출력은 마치 마법처럼 느껴질 수 있습니다. 마치 모델 안에 지능적인 연산자가 앉아서 모든 것을 조종하는 것처럼 말입니다. 하지만 이러한 모델은 근본적으로 보간적이라는 점을 기억하는 것이 중요합니다. 어텐션 메커니즘 덕분에 모델은 영어로 작성된 모든 텍스트의 상당 부분을 차지하는 보간 임베딩 공간을 학습합니다. 이 임베딩 공간을 탐색하면 흥미롭고 예상치 못한 일반화를 얻을 수 있지만, 인간 수준의 지능에 근접하는 근본적으로 새로운 것을 합성할 수는 없습니다.

### Summary

* 언어 모델은 특정 확률 분포(p(토큰|이전 토큰))를 학습하는 모델입니다.
  - 언어 모델은 광범위하게 응용되지만, 가장 중요한 점은 루프에서 모델을 호출하여 텍스트를 생성할 수 있다는 것입니다. 즉, 한 시점의 출력 토큰이 다음 시점의 입력 토큰이 됩니다.
  - 마스크드 언어 모델은 관련 확률 분포(p(토큰|주변 토큰))를 학습하며 텍스트 및 개별 토큰 분류에 유용할 수 있습니다.
  - 시퀀스-투-시퀀스 언어 모델은 대상 시퀀스의 이전 토큰과 완전히 별개의 고정된 소스 시퀀스가 ​​모두 주어졌을 때 다음 토큰을 예측하도록 학습합니다. 시퀀스-투-시퀀스 모델은 번역 및 질의응답과 같은 문제에 유용합니다.
  - 시퀀스-투-시퀀스 모델은 일반적으로 두 개의 구성 요소로 이루어져 있습니다. 인코더는 소스 시퀀스의 표현을 계산하고, 디코더는 이 표현을 입력으로 받아 이전 토큰을 기반으로 대상 시퀀스의 다음 토큰을 예측합니다.

* 어텐션은 모델이 현재 처리 중인 토큰의 컨텍스트에 따라 시퀀스 내 어디에서든 선택적으로 정보를 가져올 수 있도록 하는 메커니즘입니다.
  - 어텐션은 텍스트에서 장거리 의존성으로 인해 RNN이 겪는 문제를 해결합니다.
  - 어텐션은 두 벡터의 내적을 계산하여 어텐션 점수를 산출하는 방식으로 작동합니다. 임베딩 공간에서 서로 가까운 벡터는 어텐션 메커니즘에서 함께 합산됩니다.
* 트랜스포머는 시퀀스 전체에 정보를 전달하는 유일한 메커니즘으로 어텐션을 사용하는 시퀀스 모델링 아키텍처입니다.
  - 트랜스포머는 어텐션과 2계층 피드포워드 네트워크를 번갈아 쌓아 올려 작동합니다.
  - 트랜스포머는 많은 매개변수와 방대한 훈련 데이터를 처리하면서도 언어 모델링 문제에서 정확도를 향상시킬 수 있습니다.
  - RNN과 달리 트랜스포머는 훈련 시 시퀀스 길이만큼의 반복문이 없어 여러 대의 컴퓨터에서 병렬로 모델을 훈련하기가 훨씬 쉽습니다.
  - 트랜스포머 인코더는 양방향 어텐션을 사용하여 시퀀스의 풍부한 표현을 구축합니다.
  - 트랜스포머 디코더는 인과적 어텐션을 사용하여 언어 모델에서 다음 단어를 예측합니다.